# Recommendations

Wraps `leonia_traffic.analysis.recommendations.generate_recommendations` plus a diversion-impact estimator. Each rule-triggered recommendation gets:

- the underlying cut-through metrics
- the hourly Visitor profile
- an equity / Bridge-OD context box
- a **diversion sanity check** that re-runs the assignment with the recommended intervention applied, flags >20 % spillover on any other Leonia street.

This is the dev companion to [reports/07_bridge_od.md](../../reports/07_bridge_od.md). Stakeholder versions live in `notebooks/stakeholder/`.

In [1]:
from pathlib import Path
import os, sys, warnings

import geopandas as gpd
import pandas as pd

REPO = Path.cwd()
while not (REPO / 'leonia_traffic').is_dir() and REPO.parent != REPO:
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

from leonia_traffic.analysis import congestion as cg, od_cutthrough as oc
from leonia_traffic.analysis.equity import equity_exposure_index
from leonia_traffic.analysis.jurisdiction import (
    annotate_in_leonia, filter_segments_to_leonia,
)
from leonia_traffic.analysis.recommendations import (
    Recommendation, generate_recommendations, recommendations_to_markdown,
)
from leonia_traffic.data.bridge_od_loader import (
    load_bridge_attributes, load_bridge_od,
)
from leonia_traffic.data.congestion_loader import (
    load_congestion, load_congestion_zones, summarize_link_reliability,
)
print('imports OK')

imports OK


## 1. Assemble the rule-engine inputs

In [2]:
od_df = load_bridge_od()
attr_df = load_bridge_attributes()
cdf = load_congestion()
czones = load_congestion_zones()

peak_imbalance = oc.gateway_peak_imbalance(od_df)
circuity = oc.cutthrough_index_from_circuity(attr_df)
exposure = equity_exposure_index(attr_df)

summary = summarize_link_reliability(cdf)
delay = cg.delay_hotspot_ranking(cdf)
summary = annotate_in_leonia(summary, czones)
delay = annotate_in_leonia(delay, czones)
summary_leonia = filter_segments_to_leonia(summary, czones)
delay_leonia = filter_segments_to_leonia(delay, czones)

per_street_path = Path('data/processed/leonia_streets_cutthrough_index.parquet')
per_street_df = (
    pd.read_parquet(per_street_path) if per_street_path.exists() else None
)
print(
    f'inputs: peak_imbalance={len(peak_imbalance)}, circuity={len(circuity)}, '
    f'summary_leonia={len(summary_leonia)}, delay_leonia={len(delay_leonia)}, '
    f'exposure={len(exposure)}, per_street='
    + (f'{len(per_street_df)}' if per_street_df is not None else 'absent')
)

inputs: peak_imbalance=5, circuity=5, summary_leonia=32, delay_leonia=6, exposure=5, per_street=136


## 2. Run the rule engine

In [3]:
recs = generate_recommendations(
    peak_imbalance_df=peak_imbalance,
    circuity_df=circuity,
    delay_df=delay_leonia,
    summary_df=summary_leonia,
    exposure_df=exposure,
    per_street_df=per_street_df,
)
print(f'{len(recs)} recommendations triggered')
rec_df = pd.DataFrame([
    {
        'rank': r.rank, 'severity': r.severity, 'target': r.target,
        'rule': r.rule, 'rationale': r.rationale, **r.metrics,
    }
    for r in recs
])
rec_df.head(40)

27 recommendations triggered


,rank,severity,target,rule,rationale,peak_am_weekday,weekend_peak_am,ratio,osm_way,index,wd_vol,non_local,wd_to_sat,tti,buffer,vhd,circuity_idx,trips,speed_share,fb,el,lo_inc,no_veh,renter,peak_vol,weekday_vhd,worst_tti
0,1,high,Fort Lee Road,primary_mitigation_candidate,Peak-AM weekday volume of 694 trips/day and we...,693.0,68.0,10.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,high,Willow Tree Road,residential_cutthrough_candidate,"Composite cut-through index 0.59 (1,257 weekda...",NaN,NaN,NaN,3356462.0,0.59,1257.0,0.61,3.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,high,Broad Avenue,residential_cutthrough_candidate,"Composite cut-through index 0.56 (12,762 weekd...",NaN,NaN,NaN,10030557.0,0.56,12762.0,0.65,1.26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,high,Schor Avenue,residential_cutthrough_candidate,Composite cut-through index 0.55 (740 weekday ...,NaN,NaN,NaN,17834065.0,0.55,740.0,0.52,2.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,high,Pine Hill Road,residential_cutthrough_candidate,Composite cut-through index 0.51 (780 weekday ...,NaN,NaN,NaN,532511.0,0.51,780.0,0.63,1.41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,high,Main Street,residential_cutthrough_candidate,"Composite cut-through index 0.51 (34,644 weekd...",NaN,NaN,NaN,11099916.0,0.51,34644.0,0.63,1.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,high,Nordhoff Drive,residential_cutthrough_candidate,Composite cut-through index 0.48 (919 weekday ...,NaN,NaN,NaN,8998330.0,0.48,919.0,0.58,1.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,high,Fort Lee Road,failing_corridor_exclude_from_diversion,Worst-hour TTI 2.01 and Buffer Index 8.81 show...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.01,8.81,71.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,medium,Grand Avenue,high_circuity_detour_evidence,"89% of Peak-AM trips have circuity > 2, meanin...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.893,250.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,medium,Grand Avenue,high_circuity_detour_evidence,"46% of Peak-AM trips have circuity > 2, meanin...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.460,167.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Enrich residential recs with hourly profile + cut-through context

For every recommendation that names a specific OSM way, attach the worst hour, peak ratio, non-local-home share, and Visitor share.

In [4]:
hp = pd.read_parquet('data/processed/derived/hourly_profiles.parquet')
ct = pd.read_parquet('data/processed/derived/cutthrough_index.parquet')
hour_cols = [f'h{i:02d}' for i in range(24)]

def _enrich(rec: Recommendation) -> dict:
    osm_id = rec.metrics.get('osm_way_id') or rec.metrics.get('osm_id')
    out = {
        'rank': rec.rank, 'severity': rec.severity,
        'target': rec.target, 'rule': rec.rule,
    }
    if osm_id is None:
        return out
    ctx_ct = ct[ct['osm_way_id'] == osm_id]
    ctx_hp = hp[hp['osm_way_id'] == osm_id]
    if not ctx_ct.empty:
        row = ctx_ct.iloc[0]
        out.update({
            'cutthrough_index': float(row['cutthrough_index']),
            'non_local_home_share': float(row.get('non_local_home_share',
                                                    float('nan'))),
            'speeding_share': float(row.get('speeding_share', float('nan'))),
            'thursday_volume': float(row.get('thursday_volume',
                                              float('nan'))),
        })
    if not ctx_hp.empty:
        hours = ctx_hp.iloc[0][hour_cols].astype(float)
        out['peak_hour'] = int(hours.idxmax()[1:])
        out['peak_volume'] = float(hours.max())
        out['off_peak_volume'] = float(hours.median())
    return out

enriched = pd.DataFrame([_enrich(r) for r in recs])
enriched.head(30)

,rank,severity,target,rule
0,1,high,Fort Lee Road,primary_mitigation_candidate
1,2,high,Willow Tree Road,residential_cutthrough_candidate
2,3,high,Broad Avenue,residential_cutthrough_candidate
3,4,high,Schor Avenue,residential_cutthrough_candidate
4,5,high,Pine Hill Road,residential_cutthrough_candidate
5,6,high,Main Street,residential_cutthrough_candidate
6,7,high,Nordhoff Drive,residential_cutthrough_candidate
7,8,high,Fort Lee Road,failing_corridor_exclude_from_diversion
8,9,medium,Grand Avenue,high_circuity_detour_evidence
9,10,medium,Grand Avenue,high_circuity_detour_evidence


## 4. Diversion sanity check

For every high/medium recommendation that targets a specific OSM way, simulate the proposed intervention (assume Closure as the conservative bound) and surface streets that absorb > 20 % spillover.

In [5]:
from leonia_traffic.network.osm_builder import build_or_load_network
from leonia_traffic.assignment import (
    apply_scenarios_to_graph, bridge_od_to_demand, build_assignment_graph,
    run_ue,
)
from leonia_traffic.simulation.scenarios import Closure

CANON = Path('data/processed/streetlight')
nodes, links, _ = build_or_load_network()
G_base = build_assignment_graph(nodes, links)
bod = pd.read_parquet(CANON / 'bridge_od.parquet')
zones = gpd.read_parquet(CANON / 'bridge_od_zones.parquet')
demand = bridge_od_to_demand(bod, zones, G_base, day_type_code=1, day_part_code=2)
base_res = run_ue(G_base, demand, max_iter=25)
base_by_way = base_res.by_osm_way().set_index('osm_way_id')['assigned_volume_vph']
print(f'baseline UE: {base_res.n_iterations} iters, gap={base_res.final_gap:.2e}')

baseline UE: 1 iters, gap=0.00e+00


In [6]:
DIVERSION_THRESHOLD_PCT = 20.0
candidate_ids = [
    int(r.metrics['osm_way_id'])
    for r in recs
    if r.severity in ('high', 'medium') and r.metrics.get('osm_way_id')
]
candidate_ids = list(dict.fromkeys(candidate_ids))[:10]   # de-dup & cap
print(f'Testing diversion for {len(candidate_ids)} candidate streets:')

rows = []
for osm_id in candidate_ids:
    sc = Closure(name=f'closure-{osm_id}', osm_way_ids=[osm_id])
    new_nodes, new_links = apply_scenarios_to_graph(nodes, links, [sc])
    G_sc = build_assignment_graph(new_nodes, new_links)
    sc_res = run_ue(G_sc, demand, max_iter=25)
    sc_by_way = sc_res.by_osm_way().set_index('osm_way_id')['assigned_volume_vph']
    combined = pd.concat([
        base_by_way.rename('base'), sc_by_way.rename('scen'),
    ], axis=1).fillna(0.0)
    combined['delta_vph'] = combined['scen'] - combined['base']
    combined['pct'] = (combined['delta_vph']
                       / combined['base'].replace(0, pd.NA)) * 100
    spillover = combined[(combined.index != osm_id)
                          & (combined['pct'].abs() > DIVERSION_THRESHOLD_PCT)
                          & (combined['delta_vph'].abs() > 5)]
    spillover = spillover.sort_values('delta_vph', ascending=False)
    rows.append({
        'closed_osm_way': osm_id,
        'closed_baseline_vph': float(base_by_way.get(osm_id, 0.0)),
        'n_spillover_streets': int(len(spillover)),
        'max_pct_increase': float(spillover['pct'].max()) if len(spillover) else 0.0,
        'top_spillover_ways': spillover.head(3).index.tolist(),
    })
diversion_df = pd.DataFrame(rows)
diversion_df

Testing diversion for 0 candidate streets:


""


## 5. Markdown export

Same table that script 07 writes into `reports/07_bridge_od.md`.

In [7]:
from IPython.display import Markdown
Markdown(recommendations_to_markdown(recs))

| # | Severity | Target | Rule | Rationale | Metrics |
|---|----------|--------|------|-----------|---------|
| 1 | HIGH | Fort Lee Road | primary_mitigation_candidate | Peak-AM weekday volume of 694 trips/day and weekday/weekend ratio of 10.1× indicate concentrated commuter cut-through; this is the primary candidate for traffic-calming or routing changes. | peak_am_weekday=693, weekend_peak_am=68, ratio=10.1 |
| 2 | HIGH | Willow Tree Road | residential_cutthrough_candidate | Composite cut-through index 0.59 (1,257 weekday Visitor trips/day, 61% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate. | osm_way=3356462, index=0.59, wd_vol=1257, non_local=0.61, wd_to_sat=3.05 |
| 3 | HIGH | Broad Avenue | residential_cutthrough_candidate | Composite cut-through index 0.56 (12,762 weekday Visitor trips/day, 65% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate. | osm_way=10030557, index=0.56, wd_vol=12762, non_local=0.65, wd_to_sat=1.26 |
| 4 | HIGH | Schor Avenue | residential_cutthrough_candidate | Composite cut-through index 0.55 (740 weekday Visitor trips/day, 52% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate. | osm_way=17834065, index=0.55, wd_vol=740, non_local=0.52, wd_to_sat=2.88 |
| 5 | HIGH | Pine Hill Road | residential_cutthrough_candidate | Composite cut-through index 0.51 (780 weekday Visitor trips/day, 63% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate. | osm_way=532511, index=0.51, wd_vol=780, non_local=0.63, wd_to_sat=1.41 |
| 6 | HIGH | Main Street | residential_cutthrough_candidate | Composite cut-through index 0.51 (34,644 weekday Visitor trips/day, 63% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate. | osm_way=11099916, index=0.51, wd_vol=34644, non_local=0.63, wd_to_sat=1.07 |
| 7 | HIGH | Nordhoff Drive | residential_cutthrough_candidate | Composite cut-through index 0.48 (919 weekday Visitor trips/day, 58% with home ≥3 mi away). Residential street with strong direct evidence of regional pass-through traffic; primary calming candidate. | osm_way=8998330, index=0.48, wd_vol=919, non_local=0.58, wd_to_sat=1.45 |
| 8 | HIGH | Fort Lee Road | failing_corridor_exclude_from_diversion | Worst-hour TTI 2.01 and Buffer Index 8.81 show this corridor is already failing; any mitigation that diverts traffic onto it would worsen conditions. | tti=2.01, buffer=8.81, vhd=71.7 |
| 9 | MEDIUM | Grand Avenue | high_circuity_detour_evidence | 89% of Peak-AM trips have circuity > 2, meaning drivers travelled materially farther than the straight-line distance — consistent with arterial-bypass behavior. | circuity_idx=0.893, trips=250 |
| 10 | MEDIUM | Grand Avenue | high_circuity_detour_evidence | 46% of Peak-AM trips have circuity > 2, meaning drivers travelled materially farther than the straight-line distance — consistent with arterial-bypass behavior. | circuity_idx=0.46, trips=167 |
| 11 | MEDIUM | Broad Avenue | residential_speeding_callout | 74% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=10030557, speed_share=0.74, wd_vol=12762 |
| 12 | MEDIUM | Pine Hill Road | residential_speeding_callout | 68% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=532511, speed_share=0.68, wd_vol=780 |
| 13 | MEDIUM | Lakeview Avenue | residential_speeding_callout | 64% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=6831752, speed_share=0.64, wd_vol=799 |
| 14 | MEDIUM | Nordhoff Drive | residential_speeding_callout | 63% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=8998330, speed_share=0.63, wd_vol=919 |
| 15 | MEDIUM | Main Street | residential_speeding_callout | 59% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=11099916, speed_share=0.59, wd_vol=34644 |
| 16 | MEDIUM | Edgewood Road | residential_speeding_callout | 57% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=590540, speed_share=0.57, wd_vol=2144 |
| 17 | MEDIUM | Lakeview Avenue | residential_speeding_callout | 54% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=3873643, speed_share=0.54, wd_vol=1029 |
| 18 | MEDIUM | Hilltop Avenue | residential_speeding_callout | 53% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=19093096, speed_share=0.53, wd_vol=265 |
| 19 | MEDIUM | Fort Lee Road | residential_speeding_callout | 53% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=590576, speed_share=0.53, wd_vol=14286 |
| 20 | MEDIUM | Schor Avenue | residential_speeding_callout | 53% of non-resident pass-through trips fall in speed bins ≥25 mph on a residential block. Speed-management treatment (signage, calming, enforcement) is warranted regardless of volume disposition. | osm_way=17834065, speed_share=0.53, wd_vol=740 |
| 21 | MEDIUM | Fort Lee Road | explicit_equity_analysis_required | Travelers using this gate include sizable shares of low_income, no_vehicle; any mitigation here should include an explicit equity impact analysis before adoption. | fb=0.3, el=0.17, lo_inc=0.31, no_veh=0.18, renter=0.45, peak_vol=3468 |
| 22 | MEDIUM | Grand Avenue | explicit_equity_analysis_required | Travelers using this gate include sizable shares of foreign_born, english_limited, low_income, no_vehicle; any mitigation here should include an explicit equity impact analysis before adoption. | fb=0.46, el=0.32, lo_inc=0.31, no_veh=0.16, renter=0.44, peak_vol=250 |
| 23 | MEDIUM | Broad Avenue | explicit_equity_analysis_required | Travelers using this gate include sizable shares of english_limited; any mitigation here should include an explicit equity impact analysis before adoption. | fb=0.34, el=0.21, lo_inc=0.3, no_veh=0.13, renter=0.37, peak_vol=228 |
| 24 | MEDIUM | Grand Avenue | explicit_equity_analysis_required | Travelers using this gate include sizable shares of no_vehicle; any mitigation here should include an explicit equity impact analysis before adoption. | fb=0.35, el=0.2, lo_inc=0.27, no_veh=0.18, renter=0.42, peak_vol=167 |
| 25 | MEDIUM | Broad Avenue | explicit_equity_analysis_required | Travelers using this gate include sizable shares of english_limited, no_vehicle; any mitigation here should include an explicit equity impact analysis before adoption. | fb=0.37, el=0.22, lo_inc=0.27, no_veh=0.15, renter=0.4, peak_vol=65 |
| 26 | INFO | Grand Avenue | high_delay_corridor | 241 vehicle-hours of weekday delay; worst-hour TTI 1.62 at 8am (8am-9am). Total time-cost-to-public is non-trivial here. | weekday_vhd=240, worst_tti=1.62 |
| 27 | INFO | Grand Avenue | high_delay_corridor | 194 vehicle-hours of weekday delay; worst-hour TTI 1.52 at 8am (8am-9am). Total time-cost-to-public is non-trivial here. | weekday_vhd=193, worst_tti=1.52 |


## Caveats

- The diversion sanity check uses **Bridge-OD-only** demand, so it under-estimates spillover from non-GWB commuter flow. Use the numbers as *lower bounds*.
- All recommendations are filtered to streets under Borough of Leonia jurisdiction (state/federal facilities — NJ Turnpike, GWB approaches, Route 46 — are excluded by upstream filters).
- The full data dictionary lives in [docs/DATA.md](../../docs/DATA.md).